# 🔬 MulCo-PlantNet — Single Image Inference & Grad-CAM Demo

Notebook này cho phép bạn kiểm tra mô hình trên **1 ảnh tuỳ ý** từ tập Test.
Mô hình sẽ **tự động tìm caption (mô tả)** tương ứng với ảnh trong thư mục `captions_LLaVA` giống hệt như notebook đánh giá mô hình (`test_fine_tuned_model.ipynb`).


## 1. Setup & Imports


In [ ]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
from torchvision import transforms
from transformers import AutoTokenizer

# Cấu hình đường dẫn root của project
current_dir = Path.cwd()
PROJECT_ROOT = current_dir
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root set to: {PROJECT_ROOT}")

from src.models.mulco import MulCoEndToEnd
# Tái sử dụng class Dataset gốc để load mapping caption y như lúc test
from src.datasets.mulco_dataset import MulCoDataset


## 2. Khai báo các module phụ trợ & Dataset


In [ ]:
idx_to_class = {
    0: 'Apple_leaf', 1: 'Apple_rust_leaf', 2: 'Apple_Scab_Leaf', 3: 'Bell_pepper_leaf',
    4: 'Bell_pepper_leaf_spot', 5: 'Blueberry_leaf', 6: 'Cherry_leaf', 7: 'Corn_Gray_leaf_spot',
    8: 'Corn_leaf_blight', 9: 'Corn_rust_leaf', 10: 'grape_leaf', 11: 'grape_leaf_black_rot',
    12: 'Peach_leaf', 13: 'Potato_leaf_early_blight', 14: 'Potato_leaf_late_blight',
    15: 'Raspberry_leaf', 16: 'Soyabean_leaf', 17: 'Squash_Powdery_mildew_leaf', 18: 'Strawberry_leaf',
    19: 'Tomato_Early_blight_leaf', 20: 'Tomato_leaf', 21: 'Tomato_leaf_bacterial_spot',
    22: 'Tomato_leaf_late_blight', 23: 'Tomato_leaf_mosaic_virus', 24: 'Tomato_leaf_yellow_virus',
    25: 'Tomato_mold_leaf', 26: 'Tomato_Septoria_leaf_spot', 27: 'Tomato_two_spotted_spider_mites_leaf'
}

class MulCoWrapper(nn.Module):
    """Wrapper để Grad-CAM có thể chạy với input là hình ảnh (fix text tokens)."""
    def __init__(self, model, input_ids, attention_mask):
        super().__init__()
        self.model = model
        self.input_ids = input_ids
        self.attention_mask = attention_mask

    def forward(self, images):
        return self.model(images, self.input_ids, self.attention_mask)

class CustomGradCAM:
    """Implement Grad-CAM dùng forward/backward hooks."""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.target_layer.register_forward_hook(self._save_activation)
        self.target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def __call__(self, x, class_idx=None):
        self.model.eval()
        self.model.zero_grad()
        output = self.model(x)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        target = output[0, class_idx]
        target.backward()

        gradients = self.gradients.detach().cpu().numpy()[0]
        activations = self.activations.detach().cpu().numpy()[0]

        weights = np.mean(gradients, axis=(1, 2))
        cam = np.zeros(activations.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * activations[i]

        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (x.shape[3], x.shape[2]))
        cam = cam - np.min(cam)
        cam = cam / (np.max(cam) + 1e-7)
        return cam, class_idx

# Load test dataset map (dùng chung cho việc tìm kiếm caption)
print("Loading Test Dataset Mapping...")
test_dir = PROJECT_ROOT / "data/processed/PlantDocSplited_depth_AUG/test"
caption_dir = PROJECT_ROOT / "data/AIDG/captions_LLaVA"
mapping_path = PROJECT_ROOT / "data/processed/PlantDocSplited_depth_AUG/global_image_caption_mapping.json"

test_dataset = MulCoDataset(
    image_root=test_dir,
    caption_root=caption_dir,
    transform=None,
    use_depth_suppressed=False,
    strict_caption_match=False,
    image_caption_mapping_path=str(mapping_path)
)
print("✅ Sẵn sàng tìm caption cho ảnh đầu vào!")


## 3. Load Model & Tokenizer


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sử dụng thiết bị: {device}")

# Load mô hình
model = MulCoEndToEnd(num_classes=28).to(device)
ckpt_path = PROJECT_ROOT / "archive/mulco_depth_aug_cb_focal_gem/best_fine_tuned_model.pth"
model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True), strict=False)
model.eval()
print(f"✅ Đã load checkpoint từ: {ckpt_path.name}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
print("✅ Đã load tokenizer (roberta-base)")

# Transform ảnh
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


## 4. Cấu hình Input
**BẠN HÃY THAY ĐỔI ĐƯỜNG DẪN ẢNH Ở CELL NÀY**


In [ ]:
# Đường dẫn tới ảnh cần test (từ tập test)
IMAGE_PATH = PROJECT_ROOT / "data/processed/PlantDocSplited_depth_AUG/test/Potato_leaf_late_blight/test_Potato leaf late blight_1.jpg"


## 5. Chạy Inference và Hiển thị Kết quả


In [ ]:
def run_single_image(image_path):
    image_path_str = str(Path(image_path).resolve())
    if not os.path.exists(image_path_str):
        print(f"❌ Lỗi: Không tìm thấy file ảnh tại {image_path_str}")
        return
        
    print(f"Đang xử lý ảnh: {Path(image_path_str).name}")
    
    # Tìm kiếm caption từ test_dataset (giống y hệt tập test_fine_tuned_model)
    found_sample = None
    for sample in test_dataset.samples:
        if str(sample['image_path'].resolve()) == image_path_str:
            found_sample = sample
            break
            
    if not found_sample:
        print(f"⚠️ Cảnh báo: Không tìm thấy mapping caption cho ảnh này trong dataset test.")
        print(f"Sẽ dùng default caption trống.")
        caption = ""
        ground_truth = "Unknown"
    else:
        caption = found_sample['text']
        ground_truth = found_sample['class_name']
        print(f"✅ Đã tự động map caption từ JSON: {found_sample['source_json']}")
        print(f"Ground Truth Class: {ground_truth}")
        
    # Chuẩn bị ảnh
    pil_img = Image.open(image_path_str).convert("RGB")
    input_tensor = transform(pil_img).unsqueeze(0).to(device)
    rgb_img = np.array(pil_img.resize((224, 224))).astype(np.float32) / 255.0
    
    # Chuẩn bị text
    tokens = tokenizer(caption, padding=True, truncation=True, max_length=256, return_tensors="pt")
    input_ids = tokens['input_ids'].to(device)
    attention_mask = tokens['attention_mask'].to(device)
    
    # Wrap model và thiết lập layer để Grad-CAM bắt tín hiệu (block cuối của fusion module)
    wrapped = MulCoWrapper(model, input_ids, attention_mask)
    target_layer = wrapped.model.fusion_blocks[-1].restormer.ffn.project_out
    
    # Chạy Grad-CAM
    cam_generator = CustomGradCAM(wrapped, target_layer)
    grayscale_cam, pred_class_idx = cam_generator(input_tensor)
    pred_class = idx_to_class[pred_class_idx]
    
    # Lấy Top-5
    with torch.no_grad():
        logits = wrapped(input_tensor)
        probs = torch.softmax(logits, dim=1)[0]
    top5_probs, top5_indices = probs.topk(5)
    
    # Overlay heatmap
    heatmap = cv2.applyColorMap(np.uint8(255 * grayscale_cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    overlay = heatmap + rgb_img
    overlay = overlay / np.max(overlay)
    
    # Hiển thị
    fig = plt.figure(figsize=(14, 5))
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1.2])
    
    ax1 = fig.add_subplot(gs[0])
    ax1.imshow((rgb_img * 255).astype(np.uint8))
    ax1.set_title("Ảnh gốc", fontweight='bold')
    ax1.axis('off')
    
    ax2 = fig.add_subplot(gs[1])
    ax2.imshow((overlay * 255).astype(np.uint8))
    ax2.set_title("Grad-CAM Heatmap", fontweight='bold')
    ax2.axis('off')
    
    ax3 = fig.add_subplot(gs[2])
    ax3.axis('off')
    
    info = []
    info.append(f"Dự đoán: {pred_class}")
    info.append(f"Độ tin cậy: {probs[pred_class_idx].item():.2%}")
    if ground_truth != "Unknown":
        icon = "✅" if pred_class == ground_truth else "❌"
        info.append(f"Thực tế:  {ground_truth} {icon}")
    info.append("\nTop-5 Predictions:")
    for i, (idx, prob) in enumerate(zip(top5_indices, top5_probs)):
        cls_name = idx_to_class[idx.item()]
        mark = "→ " if cls_name == ground_truth else "  "
        info.append(f"{mark}{i+1}. {cls_name}: {prob.item():.2%}")
        
    ax3.text(0.05, 0.95, "\n".join(info), transform=ax3.transAxes, 
             fontsize=12, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8))
             
    plt.suptitle(f"Caption: {caption[:150]}...", fontsize=10, style='italic', y=0.05)
    plt.tight_layout()
    plt.show()

# Thực thi
run_single_image(IMAGE_PATH)
